> 在线讲义：[第 29 章 · 运行时：重叠、CUDA Graph、动态形状与 persistent kernel](https://wikieden.github.io/ai-infra-from-constraints/lessons/29-runtime.html) · 本书《从约束推导 AI Infra》

# 第 29 章 · 运行时：重叠、CUDA Graph、动态形状与 persistent kernel

配套讲义：仓库里 `ai-infra/notes/lessons/29-runtime.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 例 5-10 | 两段流水 | `host_pipeline(100, 20, 5)` | 4000 / 2020 / 2005 / 505 μs |
| 第二步 例 5-11 | 复制 $2X/B$ | `graph_total_us(mib)`、`graph_breakeven_bytes()` | 27.3 / 43.7 μs；12.8 MiB |
| 第三步 例 5-12 | $P + rT$ | `group_ms`、`total_ms`、`best(r)` | 17.0 / 12.4 / 6.8 ms；65 / 90 组 |
| 第四步 persistent | | `persistent_us()` | 126 → 114 μs |

例 5-10 主机流水、例 5-11 图输入复制、例 5-12 分桶与特化、自动调优回本、persistent kernel 取自 AII 第 5 章 §5.5。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
MiB = 2**20

**函数 `host_pipeline`**：(串行, 流水) 的总时间（μs）：流水时先准备第一段，之后每 max 完成一段。

In [ ]:
def host_pipeline(n: int, host: float, dev: float) -> tuple[float, float]:
    """(串行, 流水) 的总时间（μs）：流水时先准备第一段，之后每 max 完成一段。"""
    return n * (host + dev), host + (n - 1) * max(host, dev) + dev

**函数 `graph_total_us`**：图重放总时间 = 准备 + 输入复制（读 + 写）+ 计算。

In [ ]:
def graph_total_us(nbytes: float, compute: float = 20, prep: float = 5, bw: float = 1.792e12) -> float:
    """图重放总时间 = 准备 + 输入复制（读 + 写）+ 计算。"""
    return prep + 2 * nbytes / bw * 1e6 + compute

**函数 `graph_breakeven_bytes`**：小工具。

In [ ]:
def graph_breakeven_bytes(saving_us: float = 15, bw: float = 1.792e12) -> float:
    return saving_us * 1e-6 * bw / 2

**函数 `group_ms`**：一组 FFN（每行 6HF FLOPs）的执行时间。

In [ ]:
def group_ms(rows: int, tflops: float, h: int = 4096, f: int = 12288) -> float:
    """一组 FFN（每行 6HF FLOPs）的执行时间。"""
    return rows * 6 * h * f / (tflops * 1e12) * 1e3

In [ ]:
STRATS = {"通用": (100, 5632, 100), "分桶": (400, 8192, 200), "特化": (900, 5632, 250)}

**函数 `total_ms`**：小工具。

In [ ]:
def total_ms(name: str, r: int) -> float:
    prep, rows, tf = STRATS[name]
    return prep + r * group_ms(rows, tf)

**函数 `best`**：小工具。

In [ ]:
def best(r: int) -> str:
    return min(STRATS, key=lambda s: total_ms(s, r))

**函数 `persistent_us`**：(粗粒度两次 launch, persistent 按块) 的总时间。

In [ ]:
def persistent_us(blocks: int = 8, launch: float = 5, task: float = 0.52,
                  rows: int = 64, h: int = 4096, f: int = 12288, peak: float = 503.8e12, bw: float = 1.792e12) -> tuple[float, float]:
    """(粗粒度两次 launch, persistent 按块) 的总时间。"""
    proj = 2 * rows * h * f / peak * 1e6
    act = rows * f * 4 / bw * 1e6
    coarse = 2 * launch + blocks * (proj + act)
    fine = launch + blocks * (proj + task) + act + task
    return coarse, fine

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：例 5-10，100 段")
for h, d in ((20, 20), (20, 5), (5, 5)):
    s, p = host_pipeline(100, h, d)
    print(f"  主机 {h} μs、加速器 {d} μs：串行 {s:.0f}，流水 {p:.0f} μs")

In [ ]:
print("\n第二步：例 5-11，图重放省 15 μs，但要先把输入复制到固定地址")
for mib in (2, 16):
    print(f"  {mib:2d} MiB：图执行 {graph_total_us(mib * MiB):.1f} μs（普通 40 μs）")
print(f"  临界输入 {graph_breakeven_bytes() / MiB:.1f} MiB")

In [ ]:
print("\n第三步：例 5-12，10 次 FFN 一组")
print(f"  实际 5632 行，分桶后 8192 行（多 {(8192 / 5632 - 1) * 100:.0f}%）")
for k, (prep, rows, tf) in STRATS.items():
    print(f"  {k}：准备 {prep} ms，每组 {group_ms(rows, tf):.1f} ms，70 组 {total_ms(k, 70) / 1e3:.2f} s")
ranges, cur, start = [], best(1), 1
for r in range(2, 200):
    if best(r) != cur:
        ranges.append((cur, start, r - 1))
        cur, start = best(r), r
ranges.append((cur, start, "…"))
print(f"  最省时：{ranges}")
calls = 600 / 5e-6
print(f"  调优 10 分钟、每次省 5 μs：{calls:.1e} 次；每秒 1 万次 {calls / 1e4 / 3600:.1f} 小时，每秒 100 次 {calls / 100 / 86400:.0f} 天")

In [ ]:
print("\n第四步：persistent kernel，8 块 up 投影 + SiLU")
c, f = persistent_us()
print(f"  两次 launch {c:.0f} μs → 按块 {f:.0f} μs（{c / f:.2f} 倍）")

## 3. 代码里才看得到的

1. 换手点用未取整的每组时间算：分桶比通用每组省 4.62 ms，300 / 4.62 = 64.9，所以第 65 组起分桶更省。
2. `graph_total_us` 的复制按读 + 写共 $2X$ 字节共用一份显存带宽。
3. persistent 那段最后一块激活不能藏，所以末尾要加一次 1.76 + 0.52 μs。

## 4. 自己改一改

- 给 `STRATS` 加一种"三个桶（512、1024、2048）"，准备 600 ms，算它在哪些组数最省。
- 把 `persistent_us` 的块数从 8 改成 4、16、32，任务开销开始吃掉收益的块数是多少？
- 用 `host_pipeline` 算 vLLM 那个例子：执行只占 38% 时，主机开销减半能快多少？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 厨师快了单子没快，菜还是按单子的节奏出；
> 套餐一键重下，搬了家得先把东西搬回原址；
> 均码、三档、量体裁衣：做得多就裁，做得少就穿均码；
> 做好一个递一个，签收也要时间，跨 SM 那一下省不掉。